# Week 10 - RAG (Retrieval Augmented Generation)
**Using the Gemini API instead of OpenAI**

**Author**: Muhammad Raheel Ijaz 
**Date**: 8/16/2026

This notebook follows the Lab 10 instructions, with `ChatOpenAI` replaced by `ChatGoogleGenerativeAI` (Gemini) via LangChain's Google GenAI integration.

Make sure the `company_docs/` folder (with `hr_policy.txt`, `benefits.txt`, `it_policy.txt`) sits in the same folder as this notebook before running.


## Part 1: Document Loading
### Task 1.1: Load Documents

In [8]:
import os
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

# Load environment variables (GEMINI_API_KEY)
load_dotenv()

# Load all .txt files from company_docs/ directly with plain Python
# (avoids depending on the deprecated langchain-community DirectoryLoader/TextLoader,
# which are just simple filesystem readers with no external provider anyway)
docs_folder = 'company_docs/'
documents = []

for filename in sorted(os.listdir(docs_folder)):
    if filename.endswith('.txt'):
        filepath = os.path.join(docs_folder, filename)
        with open(filepath, 'r', encoding='utf-8') as f:
            text = f.read()
        documents.append(Document(page_content=text, metadata={'source': filepath}))

print(f'Loaded {len(documents)} documents')
print(f'First doc preview: {documents[0].page_content[:200]}...')


Loaded 3 documents
First doc preview: Employee Benefits Guide

Health Insurance:
Health insurance is fully covered by the company for all full-time employees,
including medical, dental, and vision coverage. Coverage begins on the first da...


### Task 1.2: Split into Chunks

In [9]:
# Create text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # Characters per chunk
    chunk_overlap=50,     # Overlap between chunks
    length_function=len,
    separators=['\n\n', '\n', '. ', ' ', '']
)

# Split documents
chunks = text_splitter.split_documents(documents)
print(f'Split into {len(chunks)} chunks')

print('\nSample chunks:')
for i, chunk in enumerate(chunks[:3]):
    print(f'\nChunk {i+1}:')
    print(chunk.page_content)
    print(f'Length: {len(chunk.page_content)} chars')


Split into 9 chunks

Sample chunks:

Chunk 1:
Employee Benefits Guide

Health Insurance:
Health insurance is fully covered by the company for all full-time employees,
including medical, dental, and vision coverage. Coverage begins on the first day
of employment. Dependents can be added to the plan for an additional monthly
premium, partially subsidized by the company.
Length: 324 chars

Chunk 2:
401(k) Retirement Plan:
The company matches employee 401(k) contributions up to 5% of base salary.
Matching contributions vest immediately. Employees are eligible to enroll in the
401(k) plan starting from their first day of employment.

Wellness Stipend:
Employees receive a $50 monthly wellness stipend that can be used for gym
memberships, fitness classes, or mental health apps. The stipend is reimbursed
through monthly expense reports.
Length: 441 chars

Chunk 3:
Professional Development:
Employees receive an annual $1,500 budget for courses, conferences, certifications,
or books related to t

**Understanding Parameters:**
- `chunk_size`: Target size for each chunk
- `chunk_overlap`: Characters shared between chunks (preserves context across a split)
- `separators`: Tries to split at natural boundaries (paragraphs, then sentences, then words)


## Part 2: Simple Retrieval
### Task 2.1: Build Keyword Search

In [10]:
def simple_search(query, chunks, top_k=3):
    """
    Simple keyword-based search.
    Returns top_k most relevant chunks.
    """
    query_lower = query.lower()

    # Score each chunk
    scored_chunks = []
    for chunk in chunks:
        content_lower = chunk.page_content.lower()
        # Count keyword matches
        score = 0
        for word in query_lower.split():
            score += content_lower.count(word)
        if score > 0:
            scored_chunks.append((score, chunk))

    # Sort by score and return top k
    scored_chunks.sort(reverse=True, key=lambda x: x[0])
    return [chunk for score, chunk in scored_chunks[:top_k]]


# Test it
query = 'What is the vacation policy?'
relevant = simple_search(query, chunks)
print(f'Found {len(relevant)} relevant chunks:')
for i, chunk in enumerate(relevant):
    print(f'\n--- Chunk {i+1} ---')
    print(chunk.page_content)


Found 3 relevant chunks:

--- Chunk 1 ---
Employee Benefits Guide

Health Insurance:
Health insurance is fully covered by the company for all full-time employees,
including medical, dental, and vision coverage. Coverage begins on the first day
of employment. Dependents can be added to the plan for an additional monthly
premium, partially subsidized by the company.

--- Chunk 2 ---
401(k) Retirement Plan:
The company matches employee 401(k) contributions up to 5% of base salary.
Matching contributions vest immediately. Employees are eligible to enroll in the
401(k) plan starting from their first day of employment.

Wellness Stipend:
Employees receive a $50 monthly wellness stipend that can be used for gym
memberships, fitness classes, or mental health apps. The stipend is reimbursed
through monthly expense reports.

--- Chunk 3 ---
Employee Handbook - HR Policies

Vacation Policy:
All full-time employees receive 15 days of paid vacation per year. Vacation days
accrue monthly and can be 

### Task 2.2: Test Different Queries

In [11]:
# Test multiple queries
test_queries = [
    'How many vacation days do employees get?',
    'What is the remote work policy?',
    'Tell me about parental leave',
]

for query in test_queries:
    print(f'\nQuery: {query}')
    results = simple_search(query, chunks, top_k=2)
    print(f'Found {len(results)} relevant chunks')
    if results:
        print(f'Top result: {results[0].page_content[:100]}...')



Query: How many vacation days do employees get?
Found 2 relevant chunks
Top result: Employee Handbook - HR Policies

Vacation Policy:
All full-time employees receive 15 days of paid va...

Query: What is the remote work policy?
Found 2 relevant chunks
Top result: Remote Work Policy:
Employees may work remotely up to 3 days per week. Remote work requires manager
...

Query: Tell me about parental leave
Found 2 relevant chunks
Top result: Remote Work Policy:
Employees may work remotely up to 3 days per week. Remote work requires manager
...


## Part 3: RAG Pipeline
### Task 3.1: Build RAG Function

In [12]:
# Initialize Gemini via LangChain
llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    google_api_key=os.getenv('GEMINI_API_KEY'),
    temperature=0   # Deterministic for factual answers
)


def rag_query(query, chunks, top_k=3):
    """
    RAG pipeline: Retrieve -> Generate
    """
    # Step 1: Retrieve relevant chunks
    relevant_chunks = simple_search(query, chunks, top_k)

    if not relevant_chunks:
        return 'No relevant information found in documents.'

    # Step 2: Build context
    context = '\n\n---\n\n'.join([
        chunk.page_content for chunk in relevant_chunks
    ])

    # Step 3: Create prompt
    prompt = f'''You are a helpful assistant. Answer the question using ONLY the context provided below.
If the answer is not in the context, say so.

Context:
{context}

Question: {query}

Answer:'''

    # Step 4: Generate answer
    response = llm.invoke(prompt)
    return response.content


### Task 3.2: Test RAG System

In [13]:
# Test questions
questions = [
    'How many vacation days do full-time employees get?',
    'Can employees work from home?',
    'What is the parental leave policy?',
    'What is the dress code?'  # Not in docs
]

for question in questions:
    print(f'\n{"="*60}')
    print(f'Q: {question}')
    print(f'{"="*60}')
    answer = rag_query(question, chunks)
    print(f'A: {answer}')



Q: How many vacation days do full-time employees get?
A: All full-time employees receive 15 days of paid vacation per year.

Q: Can employees work from home?


ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash\nPlease retry in 58.998243232s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-Fre

**Expected Behavior:** Questions 1-3 should get accurate answers pulled from the documents. Question 4 (dress code) isn't covered in any of the sample docs, so the model should say the answer isn't in the context.


## Bonus: Compare With vs Without RAG

In [ ]:
def ask_without_rag(question):
    """
    Ask Gemini directly (no retrieved context)
    """
    messages = [
        {'role': 'system', 'content': 'You are a helpful HR assistant.'},
        {'role': 'user', 'content': question}
    ]
    response = llm.invoke(messages)
    return response.content


# Compare
question = 'How many vacation days do employees get?'

print('WITHOUT RAG:')
print(ask_without_rag(question))

print('\nWITH RAG:')
print(rag_query(question, chunks))


**Notice:** Without RAG, Gemini gives a generic, made-up-sounding answer since it has no knowledge of your company's actual policy. With RAG, the answer is specific to YOUR documents (15 days) — this is the entire point of Retrieval Augmented Generation.

---
### Next steps
Coming up: replacing this simple keyword search with real **vector embeddings and semantic search**, so the retrieval step understands meaning, not just literal word overlap.